In [ ]:
import os
import requests
import mysql.connector
import pandas as pd
from dotenv import load_dotenv

# 1. Cargar las variables de entorno del archivo .env
load_dotenv()

def text_to_sql(pregunta_usuario):
    """Envía la pregunta a Ollama con las instrucciones del esquema."""
    url = os.getenv("OLLAMA_URL", "http://ollama:11434/api/generate")
    model = os.getenv("LLM_MODEL", "llama3.2")
    
    # Este prompt le da el "contexto" de tus tablas al modelo
    system_prompt = """
    Sos un experto en bases de datos MySQL. Tu única tarea es traducir preguntas en lenguaje natural a consultas SQL válidas.
    Tus respuestas deben ser únicamente código SQL listo para ejecutar. No saludes ni des explicaciones.
    
    ESQUEMA DE LA BASE DE DATOS (MySQL):

        1. Tablas Maestras de Configuración:
        - editorial(id_editorial [PK], nombre)
        - estadoPrestamo(id_estadoPrestamo [PK], nombre) -- Ej: 'Activo', 'Devuelto', 'Vencido'
        - descripcion(id_descripcion [PK], nombre) -- Detalle o motivo de una sanción
        - tipoSocio(id_tipoSocio [PK], nombre) -- Ej: 'Estudiante', 'Docente', 'Regular'
        - tipoSancion(id_tipoSancion [PK], nombre) -- Ej: 'Suspensión', 'Multa'
        - nacionalidad(id_nacionalidad [PK], nombre)
        - genero(id_genero [PK], nombre) -- Géneros literarios (Ej: 'Ficción', 'Terror')
        - estadoFisico(id_estadoFisico [PK], nombre) -- Ej: 'Excelente', 'Dañado', 'Baja'

        2. Tablas Principales del Negocio:
        - autor(id_autor [PK], nombre, apellido)
        - libro(isbn [PK], titulo, anio_publicacion, stock_total, stock_disponible, edicion, id_editorial [FK->editorial.id_editorial])
        - socio(id_socio [PK], dni, nombre, apellido, email, fecha_alta, activo [boolean], id_nacionalidad [FK->nacionalidad.id_nacionalidad], id_tipoSocio [FK->tipoSocio.id_tipoSocio])
        - ejemplar(id_ejemplar [PK], nro_ejemplar, id_estadoFisico [FK->estadoFisico.id_estadoFisico], isbn [FK->libro.isbn])
        - prestamo(id_prestamo [PK], fecha_prestamo, fecha_vencimiento, fecha_devolucion [NULL si no se devolvió], id_socio [FK->socio.id_socio], id_ejemplar [FK->ejemplar.id_ejemplar], id_estadoPrestamo [FK->estadoPrestamo.id_estadoPrestamo])
        - sancion(id_sancion [PK], fecha_inicio, fecha_fin, id_descripcion [FK->descripcion.id_descripcion], id_tipoSancion [FK->tipoSancion.id_tipoSancion], id_socio [FK->socio.id_socio])

        3. Tablas Intermedias (Relaciones Muchos a Muchos):
        - libroAutor(isbn [PK, FK->libro.isbn], id_autor [PK, FK->autor.id_autor])
        - generoLibro(id_genero [PK, FK->genero.id_genero], isbn [PK, FK->libro.isbn])

        4. Tabla de Auditoría:
        - auditoria_prestamos(id_auditoria [PK], id_prestamo, accion, id_socio_anterior, id_socio_nuevo, id_ejemplar_anterior, id_ejemplar_nuevo, estado_anterior, estado_nuevo, usuario_bd, fecha_auditoria)

        REGLAS CRÍTICAS DE NEGOCIO PARA CONSTRUIR LAS CONSULTAS:
        1. Préstamos Activos: Un préstamo está activo y en poder del socio si `prestamo.fecha_devolucion IS NULL`.
        2. Socios Morosos / Préstamos Vencidos: Un préstamo está vencido (y el socio está en mora) si `prestamo.fecha_devolucion IS NULL` y la fecha actual es mayor a la de vencimiento (`prestamo.fecha_vencimiento < CURDATE()`).
        3. Libros y Autores: Para buscar los libros de un autor, se debe pasar OBLIGATORIAMENTE por la tabla intermedia `libroAutor`. Un libro puede tener múltiples autores.
        4. Libros y Géneros: Para filtrar libros por género, se debe pasar OBLIGATORIAMENTE por la tabla intermedia `generoLibro`.
        5. Búsqueda de Personas: Al filtrar por nombre y apellido de socios o autores, recordá usar `LIKE` o comparar ambos campos.
        6. Si la pregunta pide estadísticas mensuales o anuales, utiliza las funciones de MySQL `MONTH(columna)` y `YEAR(columna)`.
    """

    full_prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTranslate this question to SQL: {pregunta_usuario}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nSELECT"
    
    payload = {
        "model": model,
        "prompt": full_prompt,
        "stream": False,
        "options": {
            "temperature": 0.0 # Reducimos la creatividad a CERO para que sea preciso
        }
    }
    
    response = requests.post(url, json=payload)
    resultado = response.json()
    
    
    sql_generado = resultado['response'].strip()
    return sql_generado

def ejecutar_consulta(sql):
    """Se conecta a tu base de datos de Aiven y ejecuta la consulta."""
    conn = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD")
    )
    # Pandas procesa la consulta y la convierte en una tabla visual hermosa
    df = pd.read_sql(sql, conn)
    conn.close()
    return df

def preguntar_al_agente(pregunta):
    """Función principal que orquesta todo el proceso."""
    print(f"Pregunta del usuario: '{pregunta}'")
    try:
        # Paso A: Traducir a SQL usando Ollama
        sql = text_to_sql(pregunta)
        print(f"SQL Generado por el modelo:\n{sql}\n")
        
        # Paso B: Ejecutar en MySQL Aiven
        df_resultado = ejecutar_consulta(sql)
        print("Resultado de la Base de Datos:")
        return df_resultado
    except Exception as e:
        print(f"Ocurrió un error: {e}")

In [13]:
print("¡Agente BiblioIA Activado! Escribí 'salir' para terminar.")
print("-" * 60)

while True:
    pregunta = input("\nIngresá tu pregunta para la biblioteca: ")
    if pregunta.lower() in ['salir', 'exit', 'quit']:
        print("Chau")
        break
    if pregunta.strip() == "":
        continue
        
    # Llama a tu función principal
    preguntar_al_agente(pregunta)

¡Agente BiblioIA Activado! Escribí 'salir' para terminar.
------------------------------------------------------------


Pregunta del usuario: 'dame una lista de 50 socios'
SQL Generado por el modelo:
SELECT SELECT id_socio FROM socio LIMIT 50;

Ocurrió un error: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
Chau


In [11]:
import requests

try:
    # Cambiá 'llama3.2' por el modelo exacto que bajaron si usaron otro
    res = requests.post("http://ollama:11434/api/generate", 
                        json={"model": "llama3.2", "prompt": "Hola, estás vivo?", "stream": False}, 
                        timeout=60)
    print("Respuesta de Ollama:", res.json()['response'])
except Exception as e:
    print("Error al conectar con Ollama:", e)

KeyboardInterrupt: 